In [13]:
!pip install -q google-genai faiss-cpu

import faiss
import numpy as np

from google import genai
from google.genai import types
from google.colab import userdata

client=genai.Client(api_key=userdata.get('Ragproject'))

EMBEDDING_MODEL='gemini-embedding-001'
EMBEDDING_DIMENSION=768

In [14]:
def embedding(text):
  response=client.models.embed_content(
      model=EMBEDDING_MODEL,
      contents=text,
      config=types.EmbedContentConfig(
          output_dimensionality=EMBEDDING_DIMENSION
      )
  )
  return(response.embeddings[0].values)

In [23]:
books = [
    'Python Programming for Beginners',
    'Machine Learning with Scikit-Learn',
    'Introduction to Cloud Computing',
    'Database Design and SQL',
    'Deep Learning with Neural Networks',
    'Web Development with React'
]

# EMBEDDING
book_vector = np.array([embedding(book) for book in books])
print("Book vector shape:", book_vector.shape)

# NORMALIZING
book_vector = book_vector / np.linalg.norm(book_vector, axis=1, keepdims=True)
print("Normalised book vector:", book_vector.shape)


# SEARCH
def search(queries, k_top=3):

    q_vectors = np.array([embedding(q) for q in queries])
    q_vectors = q_vectors / np.linalg.norm(q_vectors, axis=1, keepdims=True)

    scores = q_vectors @ book_vector

    #finding the highest score
    top_rank = np.argsort(scores)[::-1][:k_top]

    return[
         {
             "Book": books[index],
             "Score": scores[index]
          }
          for index in top_rank
    ]


# TEST SEARCH
prompt = [
    "book for someone starting Python",
    "learn neural networks",
    "want to understand cloud technology"
]

for i, query in enumerate(prompt, 1):
    print(f"\n_________FAISS Query {i}_______\n{query}")

    for result in results:
        print(f"{result['score']:.3f} - {result['book']}")

Book vector shape: (6, 768)
Normalised book vector: (6, 768)

_________FAISS Query 1_______
book for someone starting Python
0.715 - Introduction to Cloud Computing
0.550 - Web Development with React
0.543 - Deep Learning with Neural Networks

_________FAISS Query 2_______
learn neural networks
0.715 - Introduction to Cloud Computing
0.550 - Web Development with React
0.543 - Deep Learning with Neural Networks

_________FAISS Query 3_______
want to understand cloud technology
0.715 - Introduction to Cloud Computing
0.550 - Web Development with React
0.543 - Deep Learning with Neural Networks
